In [ ]:
import os, glob
import numpy as np
import pandas as pd
from pathlib import Path

In [ ]:
# --- Configuration ---
# Project root is one level above the notebook folder.
# The notebook is in a folder like: project/notebooks/
PROJECT_ROOT = Path.cwd().parent
# Folder containing the full sampled flux distribution CSV files.
IN_DIR = PROJECT_ROOT / "distributions4piton"
# Folder where summary-statistic CSV files will be saved.
OUT_DIR = PROJECT_ROOT / "summary_stats"
# Create the output folder if it does not already exist.
OUT_DIR.mkdir(parents=True, exist_ok=True)

# --- Batch Processing ---
# Iterate through all files matching the pattern "FAC*_full.csv"
for path in sorted(glob.glob(os.path.join(IN_DIR, "FAC*_full.csv"))):
    # Extract the facility ID (e.g., "FAC01") from the filename
    facid = os.path.basename(path).replace("_full.csv", "")
    out_path = os.path.join(OUT_DIR, f"{facid}_summary.csv")

    # Optimization: Skip processing if the summary file was already created in a previous run
    if os.path.exists(out_path):
        print("skipping", facid, "(already exists)")
        continue

    print("processing", facid)

    # Load the CSV; index_col=0 assumes the first column contains reaction IDs
    df = pd.read_csv(path, index_col=0)
    
    # Convert to a NumPy array for faster vectorized calculations
    X = df.to_numpy(float)

    # --- Data Integrity Handling ---
    # Count non-NaN (valid) entries per row. 
    # ~np.isnan(X) creates a boolean mask where True = valid data.
    n_valid = np.sum(~np.isnan(X), axis=1)
    
    # Safety check: If a reaction has zero valid samples, set to 1 to avoid DivisionByZero errors
    # during the fraction calculation later.
    n_valid[n_valid == 0] = 1

    # --- Summary Statistics Generation ---
    # We use 'nan' versions of functions (e.g., np.nanmean) to ignore missing values.
    # axis=1 performs the calculation row-wise (per reaction).
    stats = {
        "mean":      np.nanmean(X, axis=1),
        "median":    np.nanmedian(X, axis=1),
        "std":       np.nanstd(X, axis=1),
        "min":       np.nanmin(X, axis=1),
        "max":       np.nanmax(X, axis=1),
        "p05":       np.nanpercentile(X, 5, axis=1),   # 5th percentile (lower bound)
        "p95":       np.nanpercentile(X, 95, axis=1),  # 95th percentile (upper bound)
        
        # Calculate the fraction of samples that are exactly zero
        # Useful for identifying sparse reactions or inactive states
        "frac_zero": np.sum(X == 0, axis=1) / n_valid, 
    }

    # Convert the results back into a structured DataFrame
    # Using the original df.index ensures we keep the Reaction IDs
    out = pd.DataFrame(stats, index=df.index)
    
    # Save the summarized results for this facility
    out.to_csv(out_path)

print("done")

In [ ]:
base_dir = Path("../summary_stats") # set path

dfs = []

for csv_file in sorted(base_dir.glob("FAC*_summary.csv")):
    fac_id = csv_file.stem.replace("_summary", "")  # FAC004

    df = pd.read_csv(csv_file, usecols=["Row", "median"])
    df["FAC"] = fac_id

    dfs.append(df)

# long format
long_df = pd.concat(dfs, ignore_index=True)

# pivot to wide matrix
wide_df = long_df.pivot(
    index="FAC",      # rows
    columns="Row",    # columns (MAR...)
    values="median"   # table values
)

# sort columns for readability
wide_df = wide_df.sort_index(axis=1)

# drop MAR columns where all values are zero (or NaN treated as zero)
wide_df = wide_df.fillna(0)
wide_df = wide_df.loc[:, (wide_df != 0).any(axis=0)]

# save
wide_df.to_csv("../summary_stats_combined/FAC_by_MAR_median_matrix.csv")

print(wide_df.head())

In [ ]:
average_data = wide_df.copy()

average_data.columns.name = None

average_data.reset_index(inplace=True)

average_data # check of the saved data

In [ ]:
base_dir = Path("../summary_stats")

dfs = []

for csv_file in sorted(base_dir.glob("FAC*_summary.csv")):
    fac_id = csv_file.stem.replace("_summary", "")  # FAC004

    df = pd.read_csv(csv_file, usecols=["Row", "median"])
    df["FAC"] = fac_id

    dfs.append(df)

# long format
long_df = pd.concat(dfs, ignore_index=True)

# pivot to wide matrix
wide_df = long_df.pivot(
    index="FAC",      # rows
    columns="Row",    # columns (MAR...)
    values="median"   # table values
)

# sort columns for readability
wide_df = wide_df.sort_index(axis=1)

# drop MAR columns where all values are zero (or NaN treated as zero)
wide_df = wide_df.fillna(0)
wide_df = wide_df.loc[:, (wide_df != 0).any(axis=0)]

wide_df.columns.name = None  # remove columns name
wide_df = wide_df.reset_index()

# save
wide_df.to_csv("../summary_stats_combined/FAC_by_MAR_median_matrix.csv")

print(wide_df.head())

In [ ]:
base_dir = Path("../summary_stats")

dfs = []

for csv_file in sorted(base_dir.glob("FAC*_summary.csv")):
    fac_id = csv_file.stem.replace("_summary", "")  # FAC004

    df = pd.read_csv(csv_file, usecols=["Row", "std"])
    df["FAC"] = fac_id

    dfs.append(df)

# long format
long_df = pd.concat(dfs, ignore_index=True)

# pivot to wide matrix
wide_df = long_df.pivot(
    index="FAC",      # rows
    columns="Row",    # columns (MAR...)
    values="std"   # table values
)

# sort columns for readability
wide_df = wide_df.sort_index(axis=1)

# drop MAR columns where all values are zero (or NaN treated as zero)
wide_df = wide_df.fillna(0)
wide_df = wide_df.loc[:, (wide_df != 0).any(axis=0)]

# save
wide_df.to_csv("../summary_stats_combined/FAC_by_MAR_std_matrix.csv")

print(wide_df.head())

In [ ]:
standard_deviation_data = pd.read_csv("../summary_stats_combined/FAC_by_MAR_std_matrix.csv")

In [ ]:
standard_deviation_data

In [ ]:
base_dir = Path("../summary_stats")

dfs = []

for csv_file in sorted(base_dir.glob("FAC*_summary.csv")):
    fac_id = csv_file.stem.replace("_summary", "")  # FAC004

    df = pd.read_csv(csv_file, usecols=["Row", "frac_zero"])
    df["FAC"] = fac_id

    dfs.append(df)

# long format
long_df = pd.concat(dfs, ignore_index=True)

# pivot to wide matrix
wide_df = long_df.pivot(
    index="FAC",      # rows
    columns="Row",    # columns (MAR...)
    values="frac_zero"   # table values
)

# sort columns for readability
wide_df = wide_df.sort_index(axis=1)

# drop MAR columns where all values are zero (or NaN treated as zero)
wide_df = wide_df.fillna(0)
wide_df = wide_df.loc[:, (wide_df < 1.0).any(axis=0)]

# save
wide_df.to_csv("../summary_stats_combined/FAC_by_MAR_frac_zero_matrix.csv")

print(wide_df.head())

In [ ]:
zero_fraction_data = pd.read_csv("../summary_stats_combined/FAC_by_MAR_frac_zero_matrix.csv")
zero_fraction_data

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# exclude FAC column
data_only = zero_fraction_data.drop(columns=["FAC"])

# count zeros per row
zeros_per_row = (data_only == 0).sum(axis=1)

zeros_sorted = np.sort(zeros_per_row.values)

q1 = np.percentile(zeros_per_row, 25)
q3 = np.percentile(zeros_per_row, 75)
iqr = q3 - q1
print(f"Q1: {q1}, Q3: {q3}, IQR: {iqr}")

plt.figure()
plt.plot(zeros_sorted)
plt.xlabel("Mice", fontsize=12)
plt.ylabel("Number of reactions", fontsize=12)
#plt.title("Number of reactions")
plt.show()

In [ ]:
# exclude FAC column
data_only = zero_fraction_data.drop(columns=["FAC"])

# count zeros per row
zeros_per_row = (data_only == 0).sum(axis=1)

# filter rows
fac_less_than_2000_zeros = zero_fraction_data.loc[
    zeros_per_row > 2900, "FAC"
]

fac_less_than_2000_zeros


In [ ]:
data_dir = Path("../distributions4piton")
files = sorted(data_dir.glob("*.csv"))

RANDOM_SEED = 88
N_SAMPLES = 100
rng = np.random.default_rng(RANDOM_SEED)

all_dfs = []

for f in files:
    # Read header only to find all sample columns
    header = pd.read_csv(f, nrows=0)
    sample_cols_all = [c for c in header.columns if c.startswith("samples")]

    # Randomly choose 100 samples
    chosen_samples = rng.choice(sample_cols_all, size=N_SAMPLES, replace=False)

    # Columns to read
    usecols = ["Row"] + list(chosen_samples)

    df = pd.read_csv(f, usecols=usecols)
    df = df.set_index("Row").T

    # Keep your original indexing logic
    df.index = [f"{idx}_{f.stem}" for idx in df.index]

    all_dfs.append(df)

final_df = pd.concat(all_dfs, axis=0)

In [ ]:
final_df.to_csv('../100_combined_samples.csv')

In [ ]:
def get_condition(data_with_conditions, row_name='Diet'):
    '''
    Extract diet information from the metadata file.

    Inputs:
        data_with_conditions : str
            Path to the metadata CSV file containing 'Diet' and 'OmicsEarTag' rows.
        row : int (unused, kept for backward compatibility)
            Placeholder parameter (can be ignored).
    
    Outputs:
        pd.DataFrame - Clean mapping of FAC_ID to Diet
    '''
    # Load metadata
    meta = pd.read_csv(data_with_conditions, index_col=0)
    
    # Extract relevant rows 
    diet_row = meta.loc[row_name]
    fac_row = meta.loc['OmicsEarTag']
    
    # Build mapping
    diet_map = pd.DataFrame({
        'FAC_ID': fac_row.values,
        row_name: diet_row.values
    })
    
    # Drop unwanted rows (header-like entries)
    diet_map = diet_map[~diet_map['FAC_ID'].isin(['Orig_ID', 'IDFemaleAgingColony'])].reset_index(drop=True)
    
    return diet_map

In [ ]:
diet_data = get_condition("../aData_S1_AllOmicsandPhenotypeData.csv")

# tSNE to visualize

In [ ]:
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
import seaborn as sns

In [ ]:
sampling_data_100 = pd.read_csv('../100_combined_samples.csv')
sampling_data_100 = sampling_data_100.loc[:, (sampling_data_100 != 0).any(axis=0)]

In [ ]:
# Extract FACXXXX from sample name
sampling_data_100["FAC"] = sampling_data_100["Unnamed: 0"].str.extract(r"(FAC\d+)")
# check
print(sampling_data_100["FAC"].value_counts().head())

In [ ]:
# Drop non-numeric columns
X = sampling_data_100.drop(columns=["Unnamed: 0", "FAC"])
# Ensure numeric
X = X.apply(pd.to_numeric)
# Scale the data
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [ ]:
diet_data = get_condition("../aData_S1_AllOmicsandPhenotypeData.csv") # load diet data

In [ ]:
sampling_data_100 = sampling_data_100.drop(columns=["Unnamed: 0"]) # change the names of columns for ease
sampling_data_100 = sampling_data_100.rename(columns={"FAC": "FAC_ID"})

In [ ]:
# build a coloring map from diet
sampling_data_100_diet = sampling_data_100[sampling_data_100["FAC_ID"].isin(diet_data["FAC_ID"])].copy()
diet_map = diet_data.set_index("FAC_ID")["Diet"]
diet_aligned = sampling_data_100_diet["FAC_ID"].map(diet_map)
color_map = {"CD": "tab:blue", "HF": "tab:red"}
color_by_diet = diet_aligned.map(color_map)

In [ ]:
diet_labels = color_by_diet.map({
    "tab:blue": "CD",
    "tab:red": "HF"
})
# turns the color mapping into diet labels

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler

def plot_tsne_by_diet(
    sampling_data_100_diet,
    color_by_diet,
    id_col="FAC_ID",
    perplexity=30,
    random_state=42,
    scale_data=True,
    figsize=(10, 8),
    alpha=0.7,
    point_size=20
):
    """
    Apply t-SNE to numeric features and plot a 2D scatter colored by diet group.

    Parameters:
    - sampling_data_100_diet: DataFrame containing features (and optionally an ID column)
    - color_by_diet: array-like with diet labels (e.g., "CD", "HF")
    - id_col: column to drop before analysis
    - perplexity: t-SNE perplexity parameter
    - random_state: seed for reproducibility
    - scale_data: whether to standardize features
    - figsize: size of the plot
    - alpha: point transparency
    - point_size: marker size in scatter plot

    Returns:
    - tsne_df: DataFrame with TSNE1, TSNE2, and diet labels
    """

    # Copy input data to avoid modifying original
    df = sampling_data_100_diet.copy()

    # Drop ID column if present (not useful for t-SNE)
    if id_col in df.columns:
        df = df.drop(columns=[id_col])

    # Keep only numeric features
    X = df.select_dtypes(include=["number"]).copy()

    # Basic validation checks
    if X.empty:
        raise ValueError("No numeric columns found for t-SNE.")

    if len(X) != len(color_by_diet):
        raise ValueError("Length of color_by_diet must match number of rows in dataframe.")

    # Optionally standardize features
    if scale_data:
        X_values = StandardScaler().fit_transform(X)
    else:
        X_values = X.values

    # Initialize and run t-SNE
    tsne = TSNE(
        n_components=2,
        perplexity=perplexity,
        random_state=random_state,
        init="pca",
        learning_rate="auto"
    )
    tsne_result = tsne.fit_transform(X_values)

    # Build dataframe with results and labels
    tsne_df = pd.DataFrame({
        "TSNE1": tsne_result[:, 0],
        "TSNE2": tsne_result[:, 1],
        "diet": pd.Series(color_by_diet).astype(str).values
    })

    # Define color mapping for diets
    color_map = {
        "CD": "tab:blue",
        "HF": "tab:red"
    }

    # Create scatter plot grouped by diet
    plt.figure(figsize=figsize)

    for diet in tsne_df["diet"].unique():
        subset = tsne_df[tsne_df["diet"] == diet]
        plt.scatter(
            subset["TSNE1"],
            subset["TSNE2"],
            c=color_map[diet],
            label=diet,
            alpha=alpha,
            s=point_size
        )

    # Axis labels and legend
    plt.xlabel("t-SNE 1", fontsize=16)
    plt.ylabel("t-SNE 2", fontsize=16)
    plt.legend(
        title="Diet",
        fontsize=14,
        title_fontsize=16,
        markerscale=2.5
    )

    # Final layout and display
    plt.tight_layout()
    plt.show()

    return tsne_df

In [ ]:
# Run t-SNE plotting function with custom transparency and point size
tsne_df = plot_tsne_by_diet(
    sampling_data_100_diet=sampling_data_100_diet,  # input feature data
    color_by_diet=diet_labels,                     # labels used for coloring
    alpha=0.2,                                     # more transparent points
    point_size=4                                   # smaller markers
)

# UMAP

In [ ]:
from umap.umap_ import UMAP # has to be imported like this 

In [ ]:
def plot_umap_by_diet(
    sampling_data_100_diet,
    color_by_diet,
    id_col="FAC_ID",
    n_neighbors=25,
    min_dist=0.4,
    random_state=42,
    n_components=2,
    metric="manhattan",
    scale_data=True,
    figsize=(10, 8),
    alpha=0.7,
    point_size=20
):
    """
    Apply UMAP to numeric features and plot a 2D scatter colored by diet group.

    Parameters:
    - sampling_data_100_diet: DataFrame with features (optionally includes ID column)
    - color_by_diet: array-like diet labels (e.g., "CD", "HF")
    - id_col: column to drop before analysis
    - n_neighbors: controls local neighborhood size
    - min_dist: controls clustering tightness
    - random_state: seed for reproducibility
    - n_components: output dimensions (typically 2)
    - metric: distance metric used by UMAP
    - scale_data: whether to standardize features
    - figsize: plot size
    - alpha: point transparency
    - point_size: marker size

    Returns:
    - umap_df: DataFrame with UMAP1, UMAP2, and diet labels
    """

    # Copy input data
    df = sampling_data_100_diet.copy()

    # Drop ID column if present
    if id_col in df.columns:
        df = df.drop(columns=[id_col])

    # Select numeric features
    X = df.select_dtypes(include=["number"]).copy()

    # Validation checks
    if X.empty:
        raise ValueError("No numeric columns found for UMAP.")

    if len(X) != len(color_by_diet):
        raise ValueError("Length of color_by_diet must match number of rows in dataframe.")

    # Optional scaling
    if scale_data:
        X_values = StandardScaler().fit_transform(X)
    else:
        X_values = X.values

    # Apply UMAP
    umap_result = UMAP(
        n_neighbors=n_neighbors,
        min_dist=min_dist,
        random_state=random_state,
        n_components=n_components,
        metric=metric,
    ).fit_transform(X_values)

    # Create result DataFrame
    umap_df = pd.DataFrame({
        "UMAP1": umap_result[:, 0],
        "UMAP2": umap_result[:, 1],
        "diet": pd.Series(color_by_diet).astype(str).values
    })

    # Color mapping
    color_map = {
        "CD": "tab:blue",
        "HF": "tab:red"
    }

    # Plot
    plt.figure(figsize=figsize)

    for diet in umap_df["diet"].unique():
        subset = umap_df[umap_df["diet"] == diet]
        plt.scatter(
            subset["UMAP1"],
            subset["UMAP2"],
            c=color_map.get(diet, "gray"),  # fallback color if missing
            label=diet,
            alpha=alpha,
            s=point_size
        )

    # Labels and legend
    plt.xlabel("UMAP 1", fontsize=16)
    plt.ylabel("UMAP 2", fontsize=16)
    plt.legend(
        title="Diet",
        fontsize=14,
        title_fontsize=16,
        markerscale=2.5
    )

    # Final layout
    plt.tight_layout()
    plt.show()

    return umap_df

In [ ]:
diet_labels = color_by_diet.map({ # make the map like before
    "tab:blue": "CD",
    "tab:red": "HF"
})

In [ ]:
# Run UMAP plotting function with lower opacity and smaller points
umap_df = plot_umap_by_diet(
    sampling_data_100_diet=sampling_data_100_diet,  # input data
    color_by_diet=diet_labels,                     # labels for coloring
    alpha=0.2,                                     # more transparent points
    point_size=4                                   # smaller markers
)